---
## Setup: Start vLLM + Onboard OpenClaw (one cell does everything)

Run the cell below **once**. It will:
1. Start vLLM serving Qwen3-4B on your AMD GPU (background)
2. Tail the startup logs live until the model is loaded
3. Onboard OpenClaw against the local vLLM endpoint
4. Activate the credential and set lean mode
5. Start the OpenClaw gateway in the background

When it finishes, open a **new terminal** and run `openclaw tui` to chat with your agent.

In [ ]:
import subprocess, time, os, re, sys
try:
    import httpx
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "httpx"])
    import httpx

# ── 1. Start vLLM in background ─────────────────────────────────────
print("▶ Starting vLLM server...")
env = {**os.environ, "VLLM_USE_TRITON_FLASH_ATTN": "0"}
vllm_proc = subprocess.Popen(
    ["vllm", "serve", "Qwen/Qwen3-4B",
     "--served-model-name", "Qwen3-4B",
     "--api-key", "abc-123",
     "--port", "8000",
     "--enable-auto-tool-choice",
     "--tool-call-parser", "hermes",
     "--trust-remote-code",
     "--max_model_len", "24272"],
    stdout=open("/tmp/vllm.log", "w"),
    stderr=subprocess.STDOUT,
    env=env,
)
print(f"  PID {vllm_proc.pid} launched, waiting for model load...\n")

# ── 2. Tail vLLM log + poll /health until ready ─────────────────────
seen = 0
ready = False
for tick in range(300):  # up to ~5 min
    time.sleep(1)
    try:
        with open("/tmp/vllm.log") as f:
            lines = f.readlines()
        new_lines = lines[seen:]
        if new_lines:
            for ln in new_lines[-5:]:  # show last 5 new lines
                print(ln.rstrip())
            seen = len(lines)
    except FileNotFoundError:
        pass
    # Check health every 5s
    if tick % 5 == 4:
        try:
            r = httpx.get("http://localhost:8000/health", timeout=2)
            if r.status_code == 200:
                ready = True
                break
        except Exception:
            pass

if not ready:
    print("\n✗ vLLM did not become healthy in 5 minutes. Check /tmp/vllm.log")
    raise SystemExit(1)

print("\n✓ vLLM is healthy!\n")

# ── 3. Onboard OpenClaw ──────────────────────────────────────────────
print("▶ Onboarding OpenClaw...")
r = subprocess.run([
    "openclaw", "onboard", "--non-interactive", "--mode", "local",
    "--auth-choice", "custom-api-key",
    "--custom-base-url", "http://localhost:8000/v1",
    "--custom-model-id", "Qwen3-4B", "--custom-provider-id", "vllm",
    "--custom-compatibility", "openai", "--custom-api-key", "abc-123",
    "--secret-input-mode", "plaintext", "--custom-image-input",
    "--skip-channels", "--skip-search", "--skip-hooks", "--skip-ui",
    "--skip-health", "--accept-risk",
], capture_output=True, text=True)
print(r.stdout.strip())
if r.stderr.strip():
    print(r.stderr.strip())

# Activate the latest credential
match = re.search(r"(vllm:setup-[\w-]+)", r.stdout + r.stderr)
if match:
    cred_id = match.group(1)
    print(f"\n▶ Activating credential {cred_id}...")
    a = subprocess.run(
        ["openclaw", "models", "auth", "activate", cred_id, "--agent", "main"],
        capture_output=True, text=True,
    )
    print(a.stdout.strip())

# ── 4. Lean mode config ─────────────────────────────────────────────
print("\n▶ Setting lean mode...")
subprocess.run(
    ["openclaw", "config", "set",
     "agents.defaults.experimental.localModelLean", "true"],
    capture_output=True, text=True,
)
print("  Config applied.")

# ── 5. Start gateway ────────────────────────────────────────────────
print("\n▶ Starting OpenClaw gateway...")
subprocess.run(["pkill", "-f", "openclaw gateway"], capture_output=True)
time.sleep(1)
subprocess.Popen(
    ["openclaw", "gateway", "run"],
    stdout=open("/root/gateway.log", "w"),
    stderr=subprocess.STDOUT,
)
time.sleep(3)
try:
    with open("/root/gateway.log") as f:
        print(f.read().strip()[-500:])
except Exception:
    pass

print("\n" + "=" * 60)
print("✓ All done! Open a terminal and run:  openclaw tui")
print("=" * 60)

### 🦞 Meet your agent (new terminal)

The gateway runs in the background now, so the only terminal you need is the one for chatting with your agent. The TUI is interactive, so it runs in its own terminal. Open a **new terminal** (`+` → Terminal) and run:

```bash
openclaw tui
```

Once it's up, say hi to your agent to start the conversation. Type this into the chat:

```text
Hi
```

This is where you'll chat with your OpenClaw agent whenever you see the 🦞 emoji.

---
## Section 2: Pull the App and Code Debug

Now let's give the agent something to work with. Ask the agent to clone the typing app and set it up.

### 🦞 Ask your OpenClaw agent
Here is an example:

```text
"Clone https://github.com/matrived/blr_workshop.git, show me the purpose of each folder and also descrive cell-by-cell on what unsloth-finetuning/grpo_game2048-finetune-unsloth/gpt_oss_20B_Reinforcement_Learning_2048_Game_BF16.ipynb file is doing each cell in 1 line and lastly what could be improve"
```

Your agent uses shell tools to clone the repo, inspect its requirements, and install the libraries. It doesn't just answer, it runs the commands for you.